# Introduction to Language Models

## What is a Language Model?

The classic definition of a language model is a **probability distribution over serquences of tokens**. Suppose we have a **vocabulary** $\mathcal{V}$ of a set of tokens.

**Autoregressive language models** are a type of language model that generates text by predicting the next token in a sequence based on the previous tokens. They are trained to maximize the likelihood of the next token given the previous tokens. The probability of  a sequence of tokens $x_1, x_2, ..., x_n$, $P(x_{1:L})$ can be expressed as:

$$
p(x_{1:L}) = \prod_{i=1}^L p(x_i|x_{1:i})
$$

this model is one where each conditional distribution $p(x_i | x_{1:i-1)$ can be computed efficiently (e.g., using a feedforward neural network).

#### From GPT: Why can an autoregressive LM compute this distribution efficiently?

Short answer: instead of modeling all possible full sentences at once, the model breaks the problem into many small next-token predictions.

- Directly modeling $p(x_{1:L})$ over all token sequences is intractable because the number of possible sequences grows exponentially.
- Autoregressive factorization rewrites this into conditional terms: $p(x_{1:L}) = \prod_{i=1}^{L} p(x_i \mid x_{1:i-1})$.
- So at step $i$, the model only solves one classification problem: "given the context, what is the probability of each next token in the vocabulary?"
- A feedforward neural network (in practice, layers inside a Transformer) computes logits for all vocabulary tokens in one forward pass, then `softmax` gives a valid probability distribution.
- Training is efficient because we compare predicted distributions with the true next tokens using cross-entropy loss, and optimize with gradient descent.

**Tiny example**

Suppose context is: `"I drink"`.

Model predicts next-token probabilities:
- `coffee`: 0.60
- `water`: 0.25
- `tea`: 0.10
- `pizza`: 0.05

If the true next token is `coffee`, training pushes probability mass toward `coffee` and away from incorrect tokens. Repeating this over massive text data teaches useful language patterns.

So "efficiently" means: each step is a standard neural-network forward pass plus softmax, and the full sequence probability is the product of these local predictions.

### Generation

To generate an entire sequecne $x_{1:L}$ from an autoregressive language model $p$, we sample one token at a time given the token generated so far.

For $i = 1, ..., L$:
$$
x_i ~ p(x_i | x_{1:i-1})^{1/T}
$$

where $T \ge 0$ is a **temperature** parameter that controls the randomness we want:
- $T = 0$ is deterministic, choose the most probable token
- $T = 1$: sample "normally" from the pure language model
- $T \to \infty$: sample from a uniform distribution over the entire vocabulary $\mathcal{v}$.

However, if we just raise the probabilities to the power of $1/T$, the distribution may not sum to 1. We can fix this by re-normalizing the distribution, the normalized version $p_T (x_i | x_{1:i-1})$ is called the **annealed** conditional probability distribution.

### Conditional Generation

we can perform conditional generation by specifying some prefix sequence $x_{1:i}$ (called a **prompt**), and sampling the rest $x_{i+1:L}$ (called the **completion**).